# Saturation day viewer

Pick a calendar day and plot each MPPT pair the same way as the notebook gallery:
**pair sum**, **expected**, **rolling ceiling**, plus **sat_moderate** / **sat_severe** markers from `saturation_flags.csv`.

Use the project venv kernel (`venv/Scripts/python.exe`).

In [ ]:
from pathlib import Path
import sys

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from ipywidgets import Dropdown, SelectMultiple, VBox, interactive_output

ROOT = Path(".").resolve()
sys.path.insert(0, str(ROOT / "sat_work" / "research"))

import bench as B
from satsim import PAIRS, fit_pair

%matplotlib inline
plt.rcParams.update({"figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3})

FLAGS_CSV = ROOT / "saturation_flags.csv"
assert FLAGS_CSV.exists(), f"Missing {FLAGS_CSV} — run sat_work/research/recommended.py"

df = B.raw_df()
fe = B.raw_fe()
ref = fe["ref"]
dayidx = fe["dayidx"]

flags = pd.read_csv(FLAGS_CSV, parse_dates=["time"]).set_index("time").sort_index()
flags = flags.reindex(df.index)

# published-style curves (same as saturation_detection.ipynb gallery)
curves = {}
for a, b in PAIRS:
    k = f"{a}_{b}"
    s = df[a] + df[b]
    act = fe["active"][a] & fe["active"][b]
    fit = fit_pair(df, s, act, ref, dayidx)
    curves[k] = {
        "s": s,
        "expected": fit["expected"],
        "cap": pd.Series(fit["cap_t"], index=df.index),
        "mod": flags[f"{k}_sat_moderate"].fillna(0).astype(bool),
        "sev": flags[f"{k}_sat_severe"].fillna(0).astype(bool),
    }

days = sorted({ts.date() for ts in df.index})
print(f"{len(df):,} rows · {days[0]} → {days[-1]} · {len(days)} days · {len(PAIRS)} pairs")

In [ ]:
def _day_slice(day):
    d0 = pd.Timestamp(day).normalize()
    return slice(d0, d0 + pd.Timedelta(days=1))


def plot_pairs_for_day(day, pairs=None):
    """One column of the gallery: all pairs for a single day."""
    pairs = list(pairs) if pairs is not None else [f"{a}_{b}" for a, b in PAIRS]
    dd = _day_slice(day)
    n = len(pairs)
    fig, axes = plt.subplots(n, 1, figsize=(12, 3.2 * n), sharex=True)
    if n == 1:
        axes = [axes]

    for ax, k in zip(axes, pairs):
        c = curves[k]
        s = c["s"].loc[dd]
        if s.empty:
            ax.set_title(f"{k} — {pd.Timestamp(day).date()} (no data)")
            continue
        capd = c["cap"].loc[dd]
        exp = c["expected"].loc[dd]
        ax.plot(s.index, s, "b", lw=1.2, label="pair sum")
        ax.plot(s.index, exp, "orange", lw=1, label="expected")
        ax.plot(s.index, capd, "r:", lw=1, label="rolling ceiling")
        for col, color, lab in [
            ("mod", "r", "sat_moderate"),
            ("sev", "purple", "sat_severe"),
        ]:
            fl = c[col].loc[dd]
            if fl.any():
                ax.scatter(s.index[fl], s[fl], c=color, s=12, zorder=5, label=lab)
        ax.set_title(f"{k} — {pd.Timestamp(day).date()}", fontsize=10)
        ax.set_ylabel("W")
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))

    axes[0].legend(fontsize=8, loc="upper left")
    axes[-1].set_xlabel("time")
    fig.tight_layout()
    plt.show()


def plot_gallery(day_list):
    """3×N gallery like the notebook figure (rows = pairs, columns = days)."""
    day_list = list(day_list)
    if not day_list:
        print("Select at least one day.")
        return
    fig, axes = plt.subplots(len(PAIRS), len(day_list), figsize=(4.5 * len(day_list), 9), squeeze=False)
    for i, (a, b) in enumerate(PAIRS):
        k = f"{a}_{b}"
        c = curves[k]
        for j, day in enumerate(day_list):
            ax = axes[i, j]
            dd = _day_slice(day)
            s = c["s"].loc[dd]
            if s.empty:
                ax.set_title(f"{k} — {day}", fontsize=9)
                continue
            capd = c["cap"].loc[dd]
            ax.plot(s.index, s, "b", lw=1.2, label="pair sum")
            ax.plot(s.index, c["expected"].loc[dd], "orange", lw=1, label="expected")
            ax.plot(s.index, capd, "r:", lw=1, label="rolling ceiling")
            for col, color, lab in [
                ("mod", "r", "sat_moderate"),
                ("sev", "purple", "sat_severe"),
            ]:
                fl = c[col].loc[dd]
                if fl.any():
                    ax.scatter(s.index[fl], s[fl], c=color, s=12, zorder=5, label=lab)
            ax.set_title(f"{k} — {pd.Timestamp(day).date()}", fontsize=9)
            ax.xaxis.set_major_formatter(mdates.DateFormatter("%H"))
            if i == 0 and j == 0:
                ax.legend(fontsize=7)
    fig.tight_layout()
    plt.show()

## Choose one day

Defaults to a heavy moderate-flag day (`2025-05-05`). Try `2026-06-16` for severe flags.

In [ ]:
mod_any = (
    flags[[f"{a}_{b}_sat_moderate" for a, b in PAIRS]]
    .fillna(0)
    .astype(bool)
    .any(axis=1)
)
by_day = mod_any.groupby(mod_any.index.normalize()).sum()
default_day = by_day.sort_values(ascending=False).index[0].date()

day_dd = Dropdown(
    options=[(d.isoformat(), d) for d in days],
    value=default_day,
    description="Day",
)


def _update_day(day):
    plot_pairs_for_day(day)


out_day = interactive_output(_update_day, {"day": day_dd})
display(day_dd, out_day)

## Multi-day gallery (optional)

Hold Ctrl/Cmd to select several days — same 3×N layout as the reference figure.

In [ ]:
suggest = [
    pd.Timestamp("2025-05-05").date(),
    pd.Timestamp("2025-05-08").date(),
    pd.Timestamp("2025-05-17").date(),
    pd.Timestamp("2026-06-16").date(),
]
suggest = [d for d in suggest if d in set(days)]

days_ms = SelectMultiple(
    options=[(d.isoformat(), d) for d in days],
    value=tuple(suggest),
    description="Days",
    rows=8,
)


def _update_gallery(day_list):
    plot_gallery(day_list)


out_gal = interactive_output(_update_gallery, {"day_list": days_ms})
display(days_ms, out_gal)

## Direct call (no widgets)

In [ ]:
# plot_pairs_for_day("2025-05-05")
# plot_gallery(["2025-05-05", "2025-05-08", "2025-05-17", "2026-06-01"])